# IMX296 quad-fisheye calibration

End-to-end record of how the 4-camera rig was calibrated on 2026-08-28/29, including
what failed and why. Run top to bottom; every cell is the command that was actually used.

**The one result that shapes everything else:** the lenses are 1.78 mm, D190/H160 on 1/3",
i.e. a genuine **>180 degree** fisheye. Three independent confirmations - the vendor spec,
our own fits (~192 deg diagonal), and OmniNxt hard-coding `fov=190` for the same module class.

That rules out the equidistant model (`pinhole-equi` / Kannala-Brandt), which projects through
`x/z` and cannot express rays at or past 90 deg incidence. It also rules out feeding these
cameras to **cuVSLAM** directly, whose only fisheye model is that same equidistant one:
virtual-stereo rectification becomes required rather than preferred.


## 0. Hardware state the recordings depend on

Checked before every session. Two of these do **not** persist across a reboot, by design -
the cameras free-run by default and external trigger is opt-in:

| | why it matters |
|---|---|
| `trigger_mode=1` | without it the STM32 keeps pulsing, the sensors ignore it, **nothing logs an error**, and the only symptoms are AE gain-hunting and frame sets that are not sets |
| `jetson_clocks` | three concurrent streams collapse without it |
| trigger pulse width | in Fast Trigger mode the pulse width *is* the exposure. Argus reports 0.521 ms; the generator was emitting **4.986 ms**. Trusting Argus puts every stamp 2.2 ms out |


In [ ]:
# on the board (ssh tx2-eth)
sudo bash -c 'echo 1 > /sys/module/imx296/parameters/trigger_mode'
sudo jetson_clocks
sudo python3 /home/nvidia/tools/j106-trigctl.py --port /dev/ttyTHS1 status
#   period_us=33333  polarity=active_high  ch1..4_exposure_us=5000  pulse_ns=4985740
#   -> exposure_us:=4986 for the capture node


## 1. Recording: eight staged bags

Four single-camera stages for intrinsics, then four **adjacent pairs** for extrinsics.

The pair order walks the rig, not the camera names: ports are c=front-left, d=front-right,
e=back-left, f=back-right, so neighbours are **c -> d -> f -> e**, i.e. cam1 -> cam2 -> cam4 -> cam3.
`cam2` and `cam3` are **diagonal** - solving that pair asks for overlap the rig does not have.

A pair frame only counts if **both** cameras see the target at the same instant: each camera
solves the board pose independently and the extrinsic is the transform between them.

```
stage 1  cam1 alone      stage 5  cam3 + cam1   (left)
stage 2  cam2 alone      stage 6  cam1 + cam2   (front)
stage 3  cam4 alone      stage 7  cam2 + cam4   (right)
stage 4  cam3 alone      stage 8  cam4 + cam3   (rear)
```


In [ ]:
# per stage, on the board; images to eMMC (125 MB/s) not the SD card (58.7 MB/s)
ros2 bag record --qos-profile-overrides-path qos.yaml -o CAM_A \
    /cam1/image_raw /cam1/frame_meta /cam2/frame_meta /cam3/frame_meta /cam4/frame_meta /imu0

# qos.yaml pins the image topics to best_effort. The capture node publishes best-effort on
# purpose (a reliable subscriber can back-pressure the Argus thread); rosbag2 picks its QoS
# from the publishers it can see when it subscribes, so if it subscribes FIRST it asks for
# RELIABLE, the match fails, and it records NOTHING while looking perfectly healthy.


### Verify the image topic, not the file size

A bag growing steadily on IMU data alone passed my first check while recording **zero images**
for 204 seconds. Count the topic that matters:


In [ ]:
import sqlite3, glob
def image_count(bagdir, topic):
    c = sqlite3.connect(glob.glob(bagdir + '/*.db3')[0])
    t = c.execute('SELECT id FROM topics WHERE name=?', (topic,)).fetchone()
    return c.execute('SELECT count(*) FROM messages WHERE topic_id=?', (t[0],)).fetchone()[0] if t else 0

image_count('CAM_A', '/cam1/image_raw')   # must be non-zero within ~10 s of starting


## 2. Decimation must key on the trigger edge, not the frame counter

The capture node publishes 1-in-N to keep the recorder in range. My first version keyed on
each camera's Argus frame *number* - and those counters start when each camera's session
starts, so they are offset between cameras. With N=3, cam1 published edges 0,3,6... while
cam3 published 2,5,8...: **a constant 66.7 ms apart, sharing no instant at all.**

Sensors synchronised to 1 us, and then the software threw away different edges from each.
It cost a whole pairwise recording - 211 seconds with zero simultaneous frames - and was
invisible in the single-camera stages.

The fix keys on the edge index derived from the frame's own SOF time, which every camera
shares to within its 1 us skew:


In [ ]:
// argus_capture_node.cpp
const int64_t period_ns = 1000000000LL / fps_;
const int64_t edge = (int64_t(ft.sof_ns) + period_ns/2) / period_ns;
send = (edge % publish_every_n_) == 0;


## 3. ROS2 -> ROS1

Kalibr is ROS1. `FrameMeta` is excluded rather than registered: Kalibr reads only images and
IMU, and the custom type would just bloat the bag. It stays in the ROS2 originals, which are
the provenance record.


In [ ]:
rosbags-convert --src CAM_A --dst ros1/CAM_A.bag \
                --exclude-msgtype bev_camera/msg/FrameMeta


## 4. Frame filter — and its limitation

Selects a coverage-balanced subset of well-conditioned views. Two reasons: hundreds of
near-identical frames cost hours and add nothing, and a fisheye needs the **periphery**,
which is what a per-cell quota targets (rather than 'seen at least once', which lets one
cell hold a single observation beside another holding a hundred).

**The limitation, learned the hard way:** this counts tags with OpenCV's ArUco. Kalibr's own
AprilGrid detector is far stricter at the periphery, so a frame this passes as '3 tags' can
reach Kalibr's initialiser with 4 corners. Raising the threshold to 8 and then 12 never fixed
the pair solves - see section 6 for what the error actually was.


In [ ]:
# scripts/calib/extract_quarterkalibr_bags.py holds the staged variant; the core selection:
import numpy as np, cv2

GRID, QUOTA = 8, 8

def select(frames, max_frames=220):
    """Greedy against the per-cell DEFICIT: a cell keeps attracting frames until it holds
    QUOTA of them, so coverage comes out even rather than merely non-empty. Ties go to the
    sharper frame, measured on the target's own bounding box - a sharp background at another
    depth says nothing about the tags."""
    need = np.full((GRID, GRID), QUOTA, int)
    picked, pool = [], list(range(len(frames)))
    while pool and len(picked) < max_frames:
        value = lambda i: (sum(min(1, need[r, c]) for (r, c) in frames[i]['cells']),
                           frames[i]['sharpness'])
        best = max(pool, key=value)
        if value(best)[0] == 0:
            break                      # every cell satisfied: stop, do not pad
        for (r, c) in frames[best]['cells']:
            need[r, c] = max(0, need[r, c] - 1)
        picked.append(best); pool.remove(best)
    return picked, need


## 5. Intrinsics

`pinhole-equi` was tried first and **diverged on every camera**, taking 2-3 hours each before
giving up, with the solver's own diagnosis: *"Optimization diverged possibly due to a bad
initialization. (Do the models fit the lenses well?)"* It does not - see the header.

`omni-radtan` (Mei) converges in 1-3 minutes on the same data.


In [ ]:
rosrun kalibr tartan_calibrate \
  --bag /data/ros1f/cam1.bag --topics /cam1/image_raw \
  --models omni-radtan --target /data/april_6x6.yaml \
  --save_dir /data/f_cam1 --dont-show-report


In [ ]:
# results, config/calib/imx296_1456x1088/
#  camera        xi      fx / fy          cx / cy        reproj px
#  cam1 (c FL)  2.172  1695.1 / 1695.3  738.4 / 561.0   0.29 / 0.28
#  cam2 (d FR)  2.111  1647.1 / 1644.7  716.1 / 550.7   0.30 / 0.36
#  cam3 (e BL)  2.156  1687.1 / 1686.8  755.1 / 550.5   0.28 / 0.33
#  cam4 (f BR)  1.908  1546.8 / 1546.1  738.4 / 528.0   0.35 / 0.40
#
# FOV check from the Mei fit: r(theta) = f*sin(theta)/(cos(theta)+xi) peaks at
# cos(theta) = -1/xi -> theta = 117 deg, r_max ~ 875 px, against image corners at ~925 px.
# The equidistant fit agreed independently: fx ~ 536 with detections to r ~ 900 px gives
# theta = 900/536 = 1.68 rad = 96 deg, i.e. ~192 deg full field.


## 6. Extrinsics — use Kalibr's detector, do not reimplement the board

Kalibr's pair solve kept dying in its intrinsics initialiser:

```
RuntimeError: DLT algorithm needs at least 6 points ... 'count' is 5
```

**This is not sparse data.** It is OpenCV's RANSAC drawing 5-point minimal subsets, which its
own DLT then rejects. I misread it as thin detections and spent hours re-filtering recordings
on that premise - tightening thresholds until the left pair fell from 66 usable frames to 29.

The second detour was worse: I reimplemented the board model and omni unprojection by hand.
The camera model round-tripped **exactly** (0.0000 deg over the whole field) and every layout
convention was searched - row/column-major, both flips, all four corner rotations, corner
reversal, and a scan over tag spacing. The residual would not go below 11 px, and the
extrinsic came out with a **1 m baseline on a 15 cm rig**.

The fix was to stop guessing at a correspondence Kalibr already defines
(`GridCalibrationTargetAprilgrid.cpp`) and call its API: same target object, same detector,
same camera geometry, and its own `T_t_c` per view.


In [ ]:
import numpy as np, rosbag, aslam_cv as acv, aslam_cameras_april as acv_april
import kalibr_common as kc
from cv_bridge import CvBridge

def detector_for(chain_yaml, grid):
    chain = kc.ConfigReader.CameraChainParameters(chain_yaml)
    cam = kc.AslamCamera.fromParameters(chain.getCameraParameters(0))
    o = acv.GridDetectorOptions(); o.filterCornerOutliers = False
    return acv.GridDetector(cam.geometry, grid, o)

tp = kc.ConfigReader.CalibrationTargetParameters('april_6x6.yaml').getTargetParams()
opts = acv_april.AprilgridOptions()
opts.minTagsForValidObs = int(max(tp['tagRows'], tp['tagCols']) + 1)   # = 7, as Kalibr does
grid = acv_april.GridCalibrationTargetAprilgrid(tp['tagRows'], tp['tagCols'],
                                                tp['tagSize'], tp['tagSpacing'], opts)

def poses(bag, topic, det):
    """stamp -> T_target_camera, keyed on the frame's OWN header stamp (never arrival time:
    matching on bag timestamps reported ZERO simultaneous pairs on a rig triggered to 1 us)."""
    out, bridge = {}, CvBridge()
    for _, msg, _ in rosbag.Bag(bag).read_messages(topics=[topic]):
        img = bridge.imgmsg_to_cv2(msg, desired_encoding='mono8')
        ok, obs = det.findTarget(acv.Time(msg.header.stamp.secs, msg.header.stamp.nsecs),
                                 np.array(img))
        if ok:
            out[msg.header.stamp.to_nsec()] = obs.T_t_c().T()
    return out

# extrinsic = inv(T_target_camB) @ T_target_camA, averaged over simultaneous views
# (rotation averaged by SVD projection of the mean matrix)


In [ ]:
# results, config/rig/rig_extrinsics_imx296.yaml
#  pair                 poses  baseline   rotation  spread
#  left  cam3 -> cam1     153   147.7 mm    90.9 deg  0.55 deg
#  front cam1 -> cam2     161   148.7 mm    90.4 deg  0.68 deg
#  right cam2 -> cam4     105   149.2 mm    92.2 deg  0.65 deg
#  rear  cam4 -> cam3     150   149.2 mm    90.1 deg  0.55 deg
#
# Four INDEPENDENT solves, from separate recordings, agreeing on baseline to 1.5 mm.
# Ring closure cam3->cam1->cam2->cam4->cam3:  4.75 deg rotation, 4.9 mm translation
# over a ~0.6 m loop (0.8%). Rotation is the weaker half: the hops sum to 363.6 deg and
# the right pair, with the fewest poses, is the outlier at 92.2.


## 7. Camera-IMU offset (Delta)

Board **fixed**, rig **moved** - Kalibr recovers Delta by comparing the motion the camera
infers from a static target against the motion the IMU measured. Rotate about all three axes,
then translate along all three.

Two problems had to be worked around:

1. **The bag held only 43 Hz of IMU.** rosbag2's single writer thread cannot carry full-rate
   images and 200 Hz IMU together, so the surplus was dropped from the publisher queue. The
   IMU node writes every sample to CSV regardless, at the data-ready edge on the same clock,
   so the input was rebuilt from that: 19717 samples at 204 Hz over the same 96.8 s window.
2. **Kalibr's own version bug** - `aopt.NoMEstimator()` called with no argument against a
   backend requiring a double. Patched in place; it is a no-op estimator, so the value is
   irrelevant.


In [ ]:
sed -i 's/NoMEstimator()/NoMEstimator(1.0)/g' $(grep -rl 'NoMEstimator()' /catkin_ws)
rosrun kalibr kalibr_calibrate_imu_camera \
  --bag /data/ros1/CAM_IMU_full.bag --cam /data/intr_cam1.yaml \
  --imu /data/imu.yaml --target /data/april_6x6.yaml --dont-show-report


In [ ]:
# timeshift_cam_imu: -0.008062 s   (the console summary rounds this to '0.0')
# residuals: 0.366 px | 0.00156 rad/s | 0.0424 m/s^2
#
# CAVEAT: -8.06 ms is larger than the stamping discipline should leave. Two unapplied terms
# point that way - the MPU-9250 DLPF group delays (gyro 2.9 ms, accel 1.88 ms, logged but
# deliberately not applied, since one correction cannot serve both paths), and half the
# measured 16.1 ms sensor readout, which is 8.05 ms. Resolve before trusting VIO scale.


## 8. What to do differently next time

| | |
|---|---|
| **Record at 30 fps, select offline** | intrinsics need no timestamps, so an MJPEG stream to the host is valid and gives the selector far more to choose from. Our ROS path capped at ~9-11 Hz on rosbag2's single writer thread |
| **Sweep the corners deliberately** | 13-21 of 64 cells per camera stayed short of quota, all peripheral - which is exactly where a >180 deg lens needs constraint |
| **Verify the image topic, not the byte count** | one bag recorded 204 s of zero images while growing steadily on IMU data |
| **Restart capture after any `nvargus-daemon` restart** | `/tmp/argus_socket` is a *file*; restarting the daemon replaces it and orphans any container that bind-mounted it. The node stays alive and silently stops delivering |
| **Read the library's source before reimplementing its conventions** | the board correspondence cost hours of guessing; `GridCalibrationTargetAprilgrid.cpp` answers it in 15 lines |
